<a href="https://colab.research.google.com/github/parleenkaur01/credit_card_fraud_detection/blob/main/Credit_Card_Fraud_Detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# =========================================================
# IMPORTING LIBRARIES
# =========================================================
import pandas as pd # Used for data manipulation and analysis (working with tabular data)
import matplotlib.pyplot as plt # Used for creating static, animated, and interactive visualizations
import seaborn as sns # Built on top of matplotlib, used to make statistical graphics more attractive
from sklearn.model_selection import train_test_split # Function to split arrays or matrices into random train and test subsets
from sklearn.preprocessing import StandardScaler # Standardizes features by removing the mean and scaling to unit variance
from sklearn.ensemble import RandomForestClassifier # A meta estimator that fits a number of decision tree classifiers
from sklearn.linear_model import LogisticRegression # A statistical model used for binary classification tasks
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score # Tools to evaluate the model's accuracy

# =========================================================
# STEP 1: LOAD AND PREPARE DATA
# =========================================================

# Load the dataset from a CSV file into a Pandas DataFrame
data = pd.read_csv("creditcard.csv")

# Print the class distribution to visualize the class imbalance.
# In fraud detection, legitimate transactions vastly outnumber fraudulent ones.
print("Class distribution (Fraud vs No Fraud in %):")
# value_counts(normalize=True) calculates the proportion, multiplying by 100 gives the percentage
print(data["fraud"].value_counts(normalize=True) * 100)

# Separate the dataset into features (inputs) and the target label (output)
# 'X' contains all columns EXCEPT 'fraud'. The axis=1 drops the column, not the row.
X = data.drop("fraud", axis=1)
# 'y' contains ONLY the 'fraud' column (0 for legitimate, 1 for fraud)
y = data["fraud"]

# Split the data into a training set (to teach the model) and a testing set (to evaluate it)
# CRITICAL LINE EXPLANATION:
# - test_size=0.2: 20% of the data goes to testing, 80% to training.
# - stratify=y: This is crucial for imbalanced datasets. It guarantees that the 80/20 split
#   maintains the exact same ratio of fraud/legitimate transactions in both the train and test sets.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Scale the features for distance-based models like Logistic Regression.
# CRITICAL LINE EXPLANATION:
# Logistic Regression calculates relationships based on numerical weights. If one feature is
# measured in millions (like income) and another in single digits (like distance), the larger
# numbers will dominate the model unfairly. StandardScaler forces all features to be on the same scale.
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train) # Learns the scale from training data and applies it
X_test_scaled = scaler.transform(X_test) # Only applies the learned scale to the testing data (prevents data leakage)


# =========================================================
# STEP 2: HELPER FUNCTION FOR PRESENTATION VISUALS
# =========================================================

# Defining a custom function to generate and display a beautiful Confusion Matrix
def plot_confusion_matrix(y_true, y_pred, model_name):
    # Calculate the confusion matrix array (True Negatives, False Positives, False Negatives, True Positives)
    cm = confusion_matrix(y_true, y_pred)

    # Set the size of the plot figure
    plt.figure(figsize=(6, 4))

    # Use seaborn to create a heatmap.
    # annot=True puts the numbers inside the squares. fmt='d' ensures they are formatted as integers.
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False,
                xticklabels=['Legitimate', 'Fraud'],
                yticklabels=['Legitimate', 'Fraud'])

    # Add labels and title for the presentation slide
    plt.title(f'Confusion Matrix: {model_name}')
    plt.xlabel('Predicted by Model')
    plt.ylabel('Actual Truth')

    # Render the plot on the screen
    plt.show()


# =========================================================
# STEP 3: MODEL 1 - LOGISTIC REGRESSION
# =========================================================
print("\n" + "="*50)
print("TRAINING MODEL 1: LOGISTIC REGRESSION")
print("="*50)

# Initialize the Logistic Regression model
# CRITICAL LINE EXPLANATION:
# - class_weight='balanced': Since fraud is rare, this tells the algorithm to heavily penalize
#   mistakes made on the minority class (fraud). It balances the mathematical importance of both classes.
lr_model = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)

# Train the model using the SCALED training data
lr_model.fit(X_train_scaled, y_train)

# Generate predictions on the testing data
lr_predictions = lr_model.predict(X_test_scaled)

# Evaluate the model
print("\nLogistic Regression - Classification Report:")
# classification_report generates precision, recall, f1-score, and support for each class
print(classification_report(y_test, lr_predictions))

# Print ROC-AUC Score. This is a single number from 0 to 1 that evaluates the model's
# ability to distinguish between classes. A score of 0.5 is random guessing; 1.0 is perfect.
print(f"ROC-AUC Score: {roc_auc_score(y_test, lr_predictions):.4f}")

# Call our custom function to show the visual confusion matrix
plot_confusion_matrix(y_test, lr_predictions, "Logistic Regression")


# =========================================================
# STEP 4: MODEL 2 - RANDOM FOREST
# =========================================================
print("\n" + "="*50)
print("TRAINING MODEL 2: RANDOM FOREST")
print("="*50)

# Initialize the Random Forest model
# CRITICAL LINE EXPLANATION:
# - class_weight='balanced': Same as above, handles the imbalanced data.
# - max_depth=10: Restricts the depth of the decision trees to prevent overfitting.
# - n_jobs=-1: Allows the model to use all CPU cores, significantly speeding up training.
rf_model = RandomForestClassifier(class_weight='balanced', max_depth=10, n_jobs=-1, random_state=42)

# Train the model. Note: Random Forests are not sensitive to scale, so we use the unscaled X_train.
rf_model.fit(X_train, y_train)

# Generate predictions on the testing data
rf_predictions = rf_model.predict(X_test)

# Evaluate the model
print("\nRandom Forest - Classification Report:")
print(classification_report(y_test, rf_predictions))

print(f"ROC-AUC Score: {roc_auc_score(y_test, rf_predictions):.4f}")

# Call our custom function to show the visual confusion matrix
plot_confusion_matrix(y_test, rf_predictions, "Random Forest")


# =========================================================
# STEP 5: FEATURE IMPORTANCE VISUALIZATION
# =========================================================
print("\n" + "="*50)
print("FEATURE IMPORTANCE (From Random Forest)")
print("="*50)

# Extract feature importances from the trained Random Forest model
# This maps the importance score (a decimal) to the corresponding column name
importances = pd.Series(rf_model.feature_importances_, index=X.columns)

# Sort them from highest to lowest and grab the top 5
top_features = importances.sort_values(ascending=False).head(5)
print("Top 5 most important variables for detecting fraud:")
print(top_features)

# Create a horizontal bar chart to visualize the top 5 features
plt.figure(figsize=(8, 4))
top_features.plot(kind='barh', color='teal') # 'barh' creates a horizontal bar chart
plt.title('Top 5 Feature Importances (Random Forest)')
plt.gca().invert_yaxis() # Invert the Y-axis so the most important feature is at the top
plt.xlabel('Importance Score')
plt.show() # Display the bar chart